##Day5

use minibatch_sgd_momentum() from day4

In [28]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [29]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """
    시그모이드를 오버플로 없이 계산한다.

    z: 임의 shape의 실수 배열. 매우 큰 음수가 들어올 수 있다.
    반환: z와 같은 shape, 모든 원소가 (0, 1) 범위.

    예: z=[0, 2] -> 약 [0.5, 0.881]
    """
    pos_mask = z>=0
    neg_mask = ~pos_mask
    result = z.astype(float)
    result[pos_mask] =  1/(1+np.exp(-result[pos_mask]))
    result[neg_mask] = np.exp(result[neg_mask])/(1+np.exp(result[neg_mask]))
    return result

def minibatch_sgd_momentum(
    X: np.ndarray, y: np.ndarray, lr: float = 0.1, momentum: float = 0.9,
    batch_size: int = 32, n_epochs: int = 50, seed: int = 0
) -> tuple[np.ndarray, list[float]]:
    """
    Day3의 미니배치 SGD에 모멘텀을 더한다.

    momentum: 위 식의 β. 0이면 Day3 함수와 같은 결과여야 한다.
    나머지 인자·반환은 minibatch_sgd_logistic과 같다.
    """
    if not np.isin(y, [0, 1]).all():
        raise ValueError("배열 y에는 0 또는 1만 포함되어야 합니다.")
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1])          # 또는 rng.normal(...)
    
    loss_list = []
    beta = momentum
    v = 0
    for j in range(n_epochs):
        idx = rng.permutation(X.shape[0]) # np.arange + shuffle 두 줄을 대체
        X_shuffled = X[idx]
        y_shuffled = y[idx]
        

        for i in range(0, X.shape[0], batch_size):
            X_batch = X_shuffled[i : i + batch_size]
            y_batch = y_shuffled[i : i + batch_size]
            p_batch = sigmoid(X_batch@w)
            dl = (X_batch.T@(p_batch-y_batch))/len(X_batch)
            v = beta*v - lr*dl
            w += v
        p = sigmoid(X@w)
        loss_list.append(-(1/X.shape[0])*(y@np.log(p)+(1-y)@np.log(1-p)))
    return (w, loss_list)


In [30]:
from datetime import datetime, timezone
import pandas as pd
import json

In [31]:
def save_model(path: str, w: np.ndarray, feature_names: list[str],
               scaler_mean: np.ndarray, scaler_scale: np.ndarray,
               metrics: dict, version: str, threshold: float) -> None:
    """
    학습 결과를 위 JSON 계약대로 한 파일에 저장한다.

    w: 학습된 계수. 절편이 w[0]에 들어 있다면 분리해서 저장한다.
    feature_names: 학습에 쓴 열 이름을 학습 시 순서 그대로.
    scaler_mean, scaler_scale: train에서 구한 열별 평균과 표준편차.
    metrics: test 평가 결과 딕셔너리.
    version: 사람이 읽을 수 있는 버전 문자열.
    """
    if len(w)-1 != len(feature_names):
        raise ValueError(f"coefficients {len(w)-1} != feature_names {len(feature_names)}")
    if not (len(scaler_mean) == len(scaler_scale) == len(feature_names)):
        raise ValueError("scaler 통계량 길이가 feature_names와 다르다")
    if not np.isfinite(w).all():
        raise ValueError("계수에 NaN/inf가 있다 — 학습이 발산했다")
    result = {}
    result["model_version"] = version
    result["train_at"] = datetime.now(timezone.utc).isoformat()
    result["feature_names"] = feature_names
    result["coefficients"] = w[1:].tolist()
    result["intercept"] = float(w[0])
    result["scaler"] = {"mean" : scaler_mean.tolist(), "scale": scaler_scale.tolist()}
    result["threshold"] = threshold
    result["metrics"] = metrics
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2,allow_nan=False, default=float)
    return None


def load_model(path: str) -> dict:
    """
    저장한 JSON을 읽어 배열 필드를 np.ndarray로 되돌린 딕셔너리를 반환한다.

    - JSON을 파싱한다.
    - REQUIRED_KEYS 중 없는 키가 있으면 ValueError. 어떤 키가 없는지 메시지에 넣는다.
    - coefficients, scaler["mean"], scaler["scale"] 를 np.ndarray(dtype=float)로 변환한다.
    - 세 배열과 feature_names 의 길이가 서로 맞는지 검증한다. 다르면 ValueError.
    - 나머지 필드는 저장된 그대로 둔다.
    반환: 위 변환이 끝난 딕셔너리.
    """
    REQUIRED_KEYS = {"model_version", "train_at", "feature_names", "coefficients",
                 "intercept", "scaler", "threshold", "metrics"}
    
    with open(path, 'r', encoding='utf-8') as f:
        result = json.load(f)
    for i in REQUIRED_KEYS:
        if i not in result:
            raise ValueError(f"{i} is not in file")
    result["coefficients"] = np.array(result["coefficients"],dtype=float)
    result["scaler"]["mean"] = np.array(result["scaler"]["mean"],dtype=float)
    result["scaler"]["scale"] = np.array(result["scaler"]["scale"],dtype=float)
    n = len(result["feature_names"])
    for key, arr in [("coefficients", result["coefficients"]),
                    ("scaler.mean", result["scaler"]["mean"]),
                    ("scaler.scale", result["scaler"]["scale"])]:
        if len(arr) != n:
            raise ValueError(f"{key} 길이 {len(arr)} != feature_names {n}")
    return result


def predict_proba_from_model(model: dict, df: "pd.DataFrame") -> np.ndarray:
    """
    로드한 모델로 DataFrame의 각 행에 대한 양성 확률을 계산한다.

    - model["feature_names"] 에 있는 열이 df 에 없으면 ValueError. 누락된 열 이름을 메시지에 넣는다.
    - df 의 열 순서가 어떻든 feature_names 순서로 재배열해서 쓴다.
    - feature_names 에 없는 df 의 여분 열은 무시한다.
    - scaler["mean"], scaler["scale"] 로 표준화한 뒤 z = X @ coefficients + intercept 를 구하고
      sigmoid 를 적용한다. (Day3 의 안정 sigmoid 를 쓴다.)
    반환: shape (len(df),) 의 확률 배열.
    """
    names = model["feature_names"]
    missing = [c for c in names if c not in df.columns]
    if missing:
        raise ValueError(f"df에 없는 열: {missing}")
    sorted_df = df[names]
    scaled = (sorted_df-model['scaler']['mean'])/model['scaler']['scale']
    z = scaled.to_numpy(dtype=float) @ model["coefficients"] + model["intercept"]
    return sigmoid(z)

    

In [32]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

d = load_breast_cancer(as_frame=True)
X, y = d.data, d.target.to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

names = list(X.columns)
mu = Xtr.to_numpy().mean(0)          # train 에서만 구한다
sd = Xtr.to_numpy().std(0)
Ztr = (Xtr.to_numpy() - mu) / sd
Xb  = np.hstack([np.ones((len(Ztr), 1)), Ztr])      # 절편 열을 앞에

w, loss_list = minibatch_sgd_momentum(Xb, ytr, lr=0.1, momentum=0.9,
                                      batch_size=32, n_epochs=300, seed=0)

# test 평가
Zte  = (Xte.to_numpy() - mu) / sd
p_te = sigmoid(Zte @ w[1:] + w[0])
yhat = (p_te >= 0.5).astype(int)
tp = ((yhat==1)&(yte==1)).sum(); fp = ((yhat==1)&(yte==0)).sum(); fn = ((yhat==0)&(yte==1)).sum()
prec, rec = tp/(tp+fp), tp/(tp+fn)
metrics = {"accuracy": float((yhat==yte).mean()), "precision": float(prec),
           "recall": float(rec), "f1": float(2*prec*rec/(prec+rec)), "n_test": int(len(yte))}
print(metrics)

{'accuracy': 0.9736842105263158, 'precision': 0.9726027397260274, 'recall': 0.9861111111111112, 'f1': 0.9793103448275863, 'n_test': 114}


C:\Users\AIT-E USER\AppData\Local\Temp\ipykernel_27236\1224965238.py:49: RuntimeWarning: divide by zero encountered in log
  loss_list.append(-(1/X.shape[0])*(y@np.log(p)+(1-y)@np.log(1-p)))
C:\Users\AIT-E USER\AppData\Local\Temp\ipykernel_27236\1224965238.py:49: RuntimeWarning: invalid value encountered in matmul
  loss_list.append(-(1/X.shape[0])*(y@np.log(p)+(1-y)@np.log(1-p)))


In [34]:
save_model("model.json", w, names, mu, sd, metrics, "v0.1", threshold=0.5)
m = load_model("model.json")

p_loaded = predict_proba_from_model(m, Xte)
print("(1) 왕복 일치:", np.allclose(p_te, p_loaded), "| 최대 차이:", np.abs(p_te-p_loaded).max())

shuffled = Xte[list(np.random.default_rng(0).permutation(names))]
print("(2) 열 순서 무관:", np.allclose(p_loaded, predict_proba_from_model(m, shuffled)))

try:
    predict_proba_from_model(m, Xte.drop(columns=names[:2]))
    print("(3) FAIL: 누락이 통과함")
except ValueError as e:
    print("(3) PASS:", e)

(1) 왕복 일치: True | 최대 차이: 3.3306690738754696e-16
(2) 열 순서 무관: True
(3) PASS: df에 없는 열: ['mean radius', 'mean texture']
